# 01 — Ingest, Convert & Register

## Purpose
Entry point for all incoming documents. Normalises all incoming documents into formats the AI pipeline can process.
Excel files (`.xlsx`, `.xls`) are the only format that requires conversion —
all other supported formats (PDF, DOCX, images) pass through unchanged. Every file that enters the raw stage gets registered here regardless of
whether it can be parsed. Assigned stable `DOC_ID`s to every file in the pipeline, writes the full
audit record to `DOCUMENTS_INGESTED`.

## What this notebook does

### Format handling
| Format | Action |
|---|---|
| `.pdf`, `.docx`, `.jpg`, `.jpeg`, `.png`, `.tif`, `.tiff` | Pipeline-ready — registered, passed to parse queue |
| `.xlsx`, `.xls` | Converted to PDF first, then both original and converted PDF registered |
| `.doc` | Registered for audit trail only — binary format, no parser available in PoC |
| Unknown formats | Logged as warning, skipped |

### Step 1 — Excel conversion
Converts new Excel files from `RAW_DOCUMENTS_STAGE/raw_files/` to PDF
before registration. Skips files whose converted PDF already exists in
`PROCESSED_DOC_STAGE/pdf_converted_from_excel/`.

**Conversion logic:**
1. Check PROCESSED_DOC_STAGE for already-converted files — skips any Excel files whose converted PDF already exists in the output folder, so it's safe to re-run
2. Discover new `.xlsx` / `.xls` files in the raw stage
3. Render each sheet separately using openpyxl + reportlab
   - Each sheet becomes its own in-memory PDF so pypdf can count exact pages
   - CJK font (STSong-Light) registered to handle Mandarin supplier content
   - Sheet title paragraph added before each sheet's content
4. Merge all per-sheet PDFs into one workbook-level PDF using pypdf
   - Sheet boundaries always align with page boundaries — no content bleed
5. Upload merged PDF to `PROCESSED_DOC_STAGE/pdf_converted_from_excel/`

**Why convert before registering:**
The converted PDF needs its own `DOC_ID` with `SOURCE_ORIGIN_DOC_ID`
pointing to the original Excel's `DOC_ID`. Running conversion first
means both files exist on stage before either is registered, so both
rows can be written in one registration pass.

### Step 2 — File discovery
Discovers all files from both stages and classifies by format:

| Source | Format group | List |
|---|---|---|
| `RAW_DOCUMENTS_STAGE/raw_files/` | `.xlsx`, `.xls`, `.doc` | `skip_files` |
| `RAW_DOCUMENTS_STAGE/raw_files/` | `.pdf`, `.docx`, images | `candidate_files` |
| `PROCESSED_DOC_STAGE/pdf_converted_from_excel/` | `.pdf` (Excel-origin) | `converted_files` |

Checks `DOCUMENTS_INGESTED` for already-registered stage paths —
existing files are skipped for registration.

### Step 3 — Registration (DOCUMENTS_INGESTED)
Three sub-steps run in order so ID dependencies resolve before they
are needed:

**3a — Skipped format files (Excel + .doc)**
Registered first so `excel_doc_id_map` is populated before converted
PDF rows need to reference Excel `DOC_ID`s via `SOURCE_ORIGIN_DOC_ID`.

Status by format:
- `.xlsx` / `.xls` → `SKIPPED_FORMAT`
- `.doc` → `UNSUPPORTED_FORMAT`

**3b — Raw parseable files**
PDF, DOCX, image files. Status → `PENDING`.

**3c — Converted PDFs**
Excel-origin PDFs. Links to original Excel via `SOURCE_ORIGIN_DOC_ID`
by stripping `_pdfconverted` suffix and looking up `excel_doc_id_map`.
Status → `PENDING`.

### Status values written
| STATUS | Meaning |
|---|---|
| `PENDING` | Registered, ready for parse |
| `SKIPPED_FORMAT` | Excel — registered for audit, not parsed directly |
| `UNSUPPORTED_FORMAT` | Binary `.doc` — registered for audit, cannot parse in PoC |

### Outputs
| Table | What is written |
|---|---|
| `INGEST.DOCUMENTS_INGESTED` | One row per file — DOC_ID, stage path, format, hash, status |
| `PROCESSING.PROCESSED_DOC_STAGE` | Converted PDFs from Excel files |

# Install & Import Packages

In [ ]:
pip install reportlab

In [ ]:
pip install openpyxl

In [ ]:
pip install pypdf

In [ ]:
import io
import os
import uuid
import pypdf
import reportlab
import openpyxl
import json
import hashlib
import pandas as pd
import pytz

from datetime import datetime, timezone
from collections import defaultdict

from snowflake.snowpark.files import SnowflakeFile
from snowflake.snowpark.context import get_active_session
from openpyxl import load_workbook
from reportlab.lib.pagesizes import A4, landscape
from reportlab.lib.units import mm
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors
from reportlab.lib.styles import ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.cidfonts import UnicodeCIDFont
pdfmetrics.registerFont(UnicodeCIDFont('STSong-Light'))

print(f"reportlab {reportlab.Version}")
print(f"pypdf     {pypdf.__version__}")
print(f"openpyxl  {openpyxl.__version__}")

session = get_active_session() 

def now_ast():
    tz = pytz.timezone('America/Halifax')
    return datetime.now(tz).isoformat()

# File Conversion (Excel -> PDF)

In [ ]:
INPUT_STAGE    = '@PERMAFROST_POC.INGEST.RAW_DOCUMENTS_STAGE'
STAGE_NAME     = 'raw_documents_stage'
INPUT_FOLDER   = 'raw_files'
PDF_STAGE      = '@PERMAFROST_POC.PROCESSING.PROCESSED_DOC_STAGE'
PDF_STAGE_NAME = 'processed_doc_stage'
OUTPUT_FOLDER  = 'pdf_converted_from_excel'
SUFFIX         = '_pdfconverted'

CJK_FONT            = 'STSong-Light'
SECTION_HEADER_FILL = '1F7A72'

cell_style = ParagraphStyle(
    'CellText', fontName=CJK_FONT, fontSize=7, leading=9
)
header_para_style = ParagraphStyle(
    'HeaderText', fontName=CJK_FONT, fontSize=9,
    leading=11, textColor=colors.white
)


#Helpers

def get_relative_path(list_name, stage_name):
    prefix = stage_name.lower() + '/'
    if list_name.lower().startswith(prefix):
        return list_name[len(prefix):]
    return list_name


def get_vertical_merge_continuation_rows(sheet):
    continuation_rows = set()
    for merged_range in sheet.merged_cells.ranges:
        if merged_range.min_row != merged_range.max_row:
            for r in range(merged_range.min_row + 1, merged_range.max_row + 1):
                continuation_rows.add(r)
    return continuation_rows


def is_section_header_row(sheet, row_idx):
    fill = sheet.cell(row=row_idx, column=1).fill
    if fill and fill.fgColor and fill.fgColor.rgb:
        return fill.fgColor.rgb[-6:].upper() == SECTION_HEADER_FILL.upper()
    return False


def render_sheet_to_elements(sheet, styles):
    elements = []
    continuation_rows = get_vertical_merge_continuation_rows(sheet)
    max_col = sheet.max_column
    segments = []
    current_body = []

    for row_idx in range(1, sheet.max_row + 1):
        if row_idx in continuation_rows:
            continue
        row_values = [
            sheet.cell(row=row_idx, column=c).value
            for c in range(1, max_col + 1)
        ]
        is_blank = all(v is None for v in row_values)

        if is_section_header_row(sheet, row_idx):
            if current_body:
                segments.append(('body', current_body))
                current_body = []
            segments.append(('header', [row_values]))
        elif is_blank:
            if current_body:
                segments.append(('body', current_body))
                current_body = []
        else:
            stripped = list(row_values)
            while stripped and stripped[-1] is None:
                stripped.pop()
            if stripped:
                current_body.append(row_values)

    if current_body:
        segments.append(('body', current_body))

    for seg_type, seg_rows in segments:
        if not seg_rows:
            continue
        max_cols   = max(len(r) for r in seg_rows)
        normalised = [
            list(r) + [None] * (max_cols - len(r)) for r in seg_rows
        ]
        style = header_para_style if seg_type == 'header' else cell_style
        para_data = [
            [
                Paragraph(
                    '' if c is None else str(c).replace('\n', '<br/>'),
                    style
                )
                for c in row
            ]
            for row in normalised
        ]
        col_width = (landscape(A4)[0] - 40 * mm) / max(max_cols, 1)
        table     = Table(para_data, colWidths=[col_width] * max_cols)
        cmds = [
            ('GRID',         (0, 0), (-1, -1), 0.4, colors.HexColor('#BDC3C7')),
            ('VALIGN',       (0, 0), (-1, -1), 'MIDDLE'),
            ('LEFTPADDING',  (0, 0), (-1, -1), 3),
            ('RIGHTPADDING', (0, 0), (-1, -1), 3),
            ('TOPPADDING',   (0, 0), (-1, -1), 2),
            ('BOTTOMPADDING',(0, 0), (-1, -1), 2),
        ]
        if seg_type == 'header':
            cmds.append((
                'BACKGROUND', (0, 0), (-1, -1),
                colors.HexColor('#' + SECTION_HEADER_FILL)
            ))
            cmds.append(('SPAN', (0, 0), (-1, 0)))
        else:
            cmds.append((
                'ROWBACKGROUNDS', (0, 0), (-1, -1),
                [colors.white, colors.HexColor('#F2F3F4')]
            ))
        table.setStyle(TableStyle(cmds))
        elements.append(table)
        elements.append(Spacer(1, 3 * mm))

    return elements


def render_sheet_to_pdf_bytes(sheet, sheet_name):
    """Renders a single sheet to PDF bytes. Returns None if empty."""
    title_style          = getSampleStyleSheet()['Heading2']
    title_style.fontName = CJK_FONT
    elements = [
        Paragraph(f'Sheet: {sheet_name}', title_style),
        Spacer(1, 4 * mm),
        *render_sheet_to_elements(sheet, getSampleStyleSheet()),
    ]
    if len(elements) <= 2:
        return None
    buffer = io.BytesIO()
    doc    = SimpleDocTemplate(
        buffer,
        pagesize=landscape(A4),
        leftMargin=20 * mm, rightMargin=20 * mm,
        topMargin=15 * mm,  bottomMargin=15 * mm,
    )
    doc.build(elements)
    buffer.seek(0)
    return buffer.read()

In [ ]:
# Check what's already been converted

pdf_stage_files   = session.sql(f"LIST {PDF_STAGE}/{OUTPUT_FOLDER}/").collect()
already_converted = set()
for row in pdf_stage_files:
    rel = get_relative_path(row['name'], PDF_STAGE_NAME)
    already_converted.add(rel.lower())

print(f"Found {len(already_converted)} already-converted PDF(s) in {OUTPUT_FOLDER}")

# Find new Excel files to convert

stage_files = session.sql(f"LIST {INPUT_STAGE}/{INPUT_FOLDER}/").collect()
excel_files = []
skipped     = []

for row in stage_files:
    rel_path = get_relative_path(row['name'], STAGE_NAME)
    if not rel_path.lower().endswith(('.xlsx', '.xls')):
        continue
    if rel_path.startswith(OUTPUT_FOLDER + '/'):
        continue

    filename_no_ext = os.path.splitext(os.path.basename(rel_path))[0]
    expected_pdf    = f"{OUTPUT_FOLDER}/{filename_no_ext}{SUFFIX}.pdf"

    if expected_pdf.lower() in already_converted:
        skipped.append(rel_path)
    else:
        excel_files.append(rel_path)

print(f"Skipping {len(skipped)} already-converted file(s):")
for f in skipped:
    print(f"  [SKIP] {f}")

print(f"\nFound {len(excel_files)} new Excel file(s) to convert:")
for f in excel_files:
    print(f"  {f}")

if not excel_files:
    print("\nNo Excels to Convert.")

else:
    # Convert each new file

    results = []

    for rel_path in excel_files:

        filename_no_ext = os.path.splitext(os.path.basename(rel_path))[0]
        output_filename = f"{filename_no_ext}{SUFFIX}.pdf"
        output_path     = f"{PDF_STAGE}/{OUTPUT_FOLDER}/{output_filename}"

        try:
            stage_file_path = f"{INPUT_STAGE}/{rel_path}"
            with SnowflakeFile.open(stage_file_path, 'rb') as f:
                workbook = load_workbook(f, data_only=True)

            # render each sheet to its own PDF bytes
            sheet_pdfs = []
            skipped_sheets = []

            for sheet_index, sheet_name in enumerate(workbook.sheetnames):
                sheet     = workbook[sheet_name]
                pdf_bytes = render_sheet_to_pdf_bytes(sheet, sheet_name)

                if pdf_bytes is None:
                    skipped_sheets.append(sheet_name)
                    print(f"    [SKIP] sheet '{sheet_name}' — empty")
                    continue

                sheet_pdfs.append((sheet_name, pdf_bytes))

            if not sheet_pdfs:
                raise ValueError("All sheets were empty — no PDF produced")

            # merge all per-sheet PDFs into one workbook PDF
            writer = pypdf.PdfWriter()
            for _, pdf_bytes in sheet_pdfs:
                reader = pypdf.PdfReader(io.BytesIO(pdf_bytes))
                for page in reader.pages:
                    writer.add_page(page)

            merged_buffer = io.BytesIO()
            writer.write(merged_buffer)
            merged_buffer.seek(0)

            # upload merged PDF to STAGE
            session.file.put_stream(
                merged_buffer,
                output_path,
                auto_compress=False,
                overwrite=True,
            )

            sheet_summary = ', '.join(
                f"'{name}'" for name, _ in sheet_pdfs
            )
            results.append({
                'file':   rel_path,
                'status': 'SUCCESS',
                'output': output_path,
                'sheets': [name for name, _ in sheet_pdfs],
                'error':  None,
            })
            print(f"  [OK]   {rel_path} -> {output_filename} "
                  f"({len(sheet_pdfs)} sheet(s): {sheet_summary})")

        except Exception as e:
            results.append({
                'file':   rel_path,
                'status': 'ERROR',
                'output': None,
                'sheets': [],
                'error':  str(e),
            })
            print(f"  [FAIL] {rel_path} -> {e}")

    # Summary

    success_count = sum(1 for r in results if r['status'] == 'SUCCESS')
    fail_count    = len(results) - success_count

    print(f"\nDone: {success_count} converted, {fail_count} failed, "
          f"{len(skipped)} skipped (already in PDF_STAGE)")

    if fail_count:
        print("\nFailed files:")
        for r in results:
            if r['status'] == 'ERROR':
                print(f"  {r['file']}: {r['error']}")

    if success_count:
        print("\nConverted files:")
        for r in results:
            if r['status'] == 'SUCCESS':
                print(f"  {r['file']} -> {r['output']}")
                for sheet in r['sheets']:
                    print(f"    sheet: '{sheet}'")

In [ ]:
ALTER STAGE PERMAFROST_POC.INGEST.RAW_DOCUMENTS_STAGE REFRESH;
ALTER STAGE PERMAFROST_POC.PROCESSING.PROCESSED_DOC_STAGE REFRESH;

# Register Files From Stage to Snowflake Tables

In [ ]:
RAW_STAGE        = '@PERMAFROST_POC.INGEST.RAW_DOCUMENTS_STAGE'
RAW_STAGE_NAME   = 'raw_documents_stage'
RAW_FOLDER       = 'raw_files'

PDF_STAGE        = '@PERMAFROST_POC.PROCESSING.PROCESSED_DOC_STAGE'
PDF_STAGE_NAME   = 'processed_doc_stage'
PDF_FOLDER       = 'pdf_converted_from_excel'

DB               = 'PERMAFROST_POC'
INGEST_SCHEMA    = 'INGEST'
PROCESSING_SCHEMA = 'PROCESSING'

#Formats AI_PARSE_DOCUMENT supports natively
PARSEABLE_RAW    = {'.pdf', '.docx', '.jpg', '.jpeg',
                    '.png', '.tif', '.tiff'}

# Formats to skip at raw stage — handled by conversion script or unsupported in PoC
SKIP_RAW         = {'.xlsx', '.xls', '.doc'}

TRANSLATE_MODEL  = 'mistral-large2'
ENGLISH_CODE     = 'en'


#Helpers

def get_relative_path(list_name, stage_name):
    prefix = stage_name.lower() + '/'
    if list_name.lower().startswith(prefix):
        return list_name[len(prefix):]
    return list_name


def sha256_bytes(file_bytes):
    return hashlib.sha256(file_bytes).hexdigest()


def now_utc():
    return datetime.now(timezone.utc).isoformat()


# Logging 
def info(msg):    print(f"INFO:    {msg}")
def warning(msg): print(f"WARNING: {msg}")
def error(msg):   print(f"ERROR:   {msg}")

In [ ]:
# Discover files from both stages
# Build a combined file list with source metadata
# Each entry: {stage_path, filename, ext, source_channel,source_origin_doc_id (for converted PDFs)}

# Raw non-Excel files
raw_stage_files = session.sql(
    f"LIST {RAW_STAGE}/{RAW_FOLDER}/"
).collect()

candidate_files  = []   # will be parsed
skip_files       = []   # will be registered but not parsed

for row in raw_stage_files:
    rel_path = get_relative_path(row['name'], RAW_STAGE_NAME)
    ext      = os.path.splitext(rel_path)[1].lower()
    filename = os.path.basename(rel_path)

    if ext in SKIP_RAW:
        skip_files.append({
            'stage_path':           f"{RAW_STAGE}/{rel_path}",
            'filename':             filename,
            'ext':                  ext,
            'source_channel':       'batch_stage',
            'source_origin_doc_id': None,
        })
        info(f"  [SKIP] {rel_path} — {ext} not parseable in PoC environment")
        continue

    if ext not in PARSEABLE_RAW:
        warning(f"  [UNKNOWN FORMAT] {rel_path} — skipping")
        continue

    candidate_files.append({
        'stage_path':            f"{RAW_STAGE}/{rel_path}",
        'filename':              filename,
        'ext':                   ext,
        'source_channel':        'batch_stage',
        'source_origin_doc_id':  None,   # not a converted file
    })

info(f"Found {len(candidate_files)} raw file(s) to process from RAW_STAGE")

# Excel-converted PDFs 
# link each converted PDF back to its original Excel DOC_ID. We handle the link by registering Excel files first,
# then converted PDFs, so we can look up the Excel DOC_ID when writing the PDF row.

pdf_stage_files = session.sql(
    f"LIST {PDF_STAGE}/{PDF_FOLDER}/"
).collect()

converted_files = []
for row in pdf_stage_files:
    rel_path = get_relative_path(row['name'], PDF_STAGE_NAME)
    filename = os.path.basename(rel_path)

    converted_files.append({
        'stage_path':   f"{PDF_STAGE}/{rel_path}",
        'filename':     filename,
        'ext':          '.pdf',
        'source_channel': 'pipeline_conversion',
        'source_origin_doc_id': None,  
    })

info(f"Found {len(converted_files)} converted PDF(s) from PDF_STAGE")

# Check which stage paths are already registered
existing = session.sql(f"""
    SELECT STAGE_PATH, DOC_ID, FILE_HASH_SHA256
    FROM {DB}.{INGEST_SCHEMA}.DOCUMENTS_INGESTED
""").collect()

existing_paths  = {row['STAGE_PATH'] for row in existing}
existing_hashes = {row['FILE_HASH_SHA256']: row['DOC_ID']
                   for row in existing if row['FILE_HASH_SHA256']}

info(f"Already registered: {len(existing_paths)} file(s) in DOCUMENTS_INGESTED")

In [ ]:

# Register files in DOCUMENTS_INGESTED
# Raw files first, then converted PDFs so Excel DOC_IDs exist before converted PDF rows reference them)
now = now_ast()
ingest_rows      = []
excel_doc_id_map = {}   # original_xlsx_basename → doc_id assigned to that Excel

# Register all skipped format files

for f in skip_files:
    stage_path = f['stage_path']
    ext        = f['ext']

    if stage_path in existing_paths:
        # Already registered — if Excel, still need DOC_ID for converted PDF link
        if ext in {'.xlsx', '.xls'}:
            existing_doc_id = next(
                r['DOC_ID'] for r in existing
                if r['STAGE_PATH'] == stage_path
            )
            basename_no_ext = os.path.splitext(f['filename'])[0].lower()
            excel_doc_id_map[basename_no_ext] = existing_doc_id
        info(f"  [ALREADY REGISTERED] {f['filename']}")
        continue

    with SnowflakeFile.open(stage_path, 'rb') as fh:
        file_bytes = fh.read()

    doc_id       = str(uuid.uuid4())
    file_hash    = sha256_bytes(file_bytes)
    duplicate_of = existing_hashes.get(file_hash)

    if ext in {'.xlsx', '.xls'}:
        status  = 'SKIPPED_FORMAT'
        basename_no_ext = os.path.splitext(f['filename'])[0].lower()
        excel_doc_id_map[basename_no_ext] = doc_id
    elif ext == '.doc':
        status = 'UNSUPPORTED_FORMAT'
    else:
        status = 'SKIPPED_FORMAT'

    ingest_rows.append({
        'DOC_ID':               doc_id,
        'PARENT_FILE_ID':       None,
        'SOURCE_ORIGIN_DOC_ID': None,
        'SOURCE_CHANNEL':       f['source_channel'],
        'SOURCE_FORMAT':        ext,
        'ORIGINAL_FILENAME':    f['filename'],
        'STAGE_PATH':           stage_path,
        'FILE_HASH_SHA256':     file_hash,
        'FILE_SIZE_BYTES':      len(file_bytes),
        'SOURCE_PAGE_START':    None,
        'SOURCE_PAGE_END':      None,
        'DUPLICATE_OF':         duplicate_of,
        'RECEIVED_AT':          now,
        'UPLOADED_BY':          'batch_ingest',
        'STATUS':               status,
    })
    info(f"  [REGISTERED] {f['filename']} → {doc_id} (STATUS: {status})")

# Register raw parseable files

parse_queue = [] 

for f in candidate_files:
    stage_path = f['stage_path']

    if stage_path in existing_paths:
        info(f"  [ALREADY REGISTERED] {f['filename']} — checking parse status")
        existing_doc_id = next(
            r['DOC_ID'] for r in existing
            if r['STAGE_PATH'] == stage_path
        )
        parse_queue.append({**f, 'doc_id': existing_doc_id})
        continue

    with SnowflakeFile.open(stage_path, 'rb') as fh:
        file_bytes = fh.read()

    doc_id    = str(uuid.uuid4())
    file_hash = sha256_bytes(file_bytes)

    duplicate_of = existing_hashes.get(file_hash)
    if duplicate_of:
        warning(f"  [HASH DUPE] {f['filename']} matches existing {duplicate_of}")

    ingest_rows.append({
        'DOC_ID':               doc_id,
        'PARENT_FILE_ID':       None,
        'SOURCE_ORIGIN_DOC_ID': None,
        'SOURCE_CHANNEL':       f['source_channel'],
        'SOURCE_FORMAT':        f['ext'],
        'ORIGINAL_FILENAME':    f['filename'],
        'STAGE_PATH':           stage_path,
        'FILE_HASH_SHA256':     file_hash,
        'FILE_SIZE_BYTES':      len(file_bytes),
        'SOURCE_PAGE_START':    None,
        'SOURCE_PAGE_END':      None,
        'DUPLICATE_OF':         duplicate_of,
        'RECEIVED_AT':          now,
        'UPLOADED_BY':          'batch_ingest',
        'STATUS':               'PENDING',
    })
    parse_queue.append({**f, 'doc_id': doc_id})
    info(f"  [REGISTERED] {f['filename']} → {doc_id}")

# Register converted PDFs
# Link each converted PDF back to its original Excel DOC_ID via
# SOURCE_ORIGIN_DOC_ID using the filename (strip _pdfconverted suffix)

SUFFIX = '_pdfconverted'

for f in converted_files:
    stage_path = f['stage_path']

    if stage_path in existing_paths:
        info(f"  [ALREADY REGISTERED] {f['filename']} — checking parse status")
        existing_doc_id = next(
            r['DOC_ID'] for r in existing
            if r['STAGE_PATH'] == stage_path
        )
        parse_queue.append({**f, 'doc_id': existing_doc_id})
        continue

    with SnowflakeFile.open(stage_path, 'rb') as fh:
        file_bytes = fh.read()

    doc_id    = str(uuid.uuid4())
    file_hash = sha256_bytes(file_bytes)

    # Derive original Excel basename to look up its DOC_ID
    pdf_stem        = os.path.splitext(f['filename'])[0]   # e.g. invoice_pdfconverted
    original_stem   = pdf_stem.replace(SUFFIX, '').lower() # e.g. invoice
    origin_doc_id   = excel_doc_id_map.get(original_stem)

    if not origin_doc_id:
        warning(f"  Could not find original Excel DOC_ID for {f['filename']} "
                f"(looked up '{original_stem}') — SOURCE_ORIGIN_DOC_ID will be NULL")

    duplicate_of = existing_hashes.get(file_hash)

    ingest_rows.append({
        'DOC_ID':               doc_id,
        'PARENT_FILE_ID':       None,
        'SOURCE_ORIGIN_DOC_ID': origin_doc_id,
        'SOURCE_CHANNEL':       'pipeline_conversion',
        'SOURCE_FORMAT':        '.pdf',
        'ORIGINAL_FILENAME':    f['filename'],
        'STAGE_PATH':           stage_path,
        'FILE_HASH_SHA256':     file_hash,
        'FILE_SIZE_BYTES':      len(file_bytes),
        'SOURCE_PAGE_START':    None,
        'SOURCE_PAGE_END':      None,
        'DUPLICATE_OF':         duplicate_of,
        'RECEIVED_AT':          now,
        'UPLOADED_BY':          'pipeline_conversion',
        'STATUS':               'PENDING',
    })
    parse_queue.append({**f, 'doc_id': doc_id})
    info(f"  [REGISTERED CONVERTED PDF] {f['filename']} → {doc_id} "
         f"(origin: {origin_doc_id})")

#write all new DOCUMENTS_INGESTED rows
if ingest_rows:
    session.write_pandas(
        pd.DataFrame(ingest_rows),
        table_name='DOCUMENTS_INGESTED',
        database=DB,
        schema=INGEST_SCHEMA,
        overwrite=False,
    )
    info(f"Wrote {len(ingest_rows)} row(s) to DOCUMENTS_INGESTED")
else:
    info("No new files to register")